# Gold Layer: Borough Summary

Aggregates Silver listing and host data to produce one row per neighbourhood.
Computes pricing, occupancy, revenue, and review metrics, then scores each neighbourhood
for three investor personas: First-Time Buyer, Experienced Landlord, Passive Income Seeker.

**Output:** `AIRBNB_INVESTMENT_DB.GOLD.BOROUGH_SUMMARY`

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
DB         = 'AIRBNB_INVESTMENT_DB'
SCHEMA_SIL = 'SILVER'
SCHEMA_GLD = 'GOLD'
TABLE_OUT  = 'BOROUGH_SUMMARY'

In [ ]:
def minmax_norm(series, invert=False):
    min_val = series.min()
    max_val = series.max()
    if max_val == min_val:
        return pd.Series([0.5] * len(series), index=series.index)
    normed = (series - min_val) / (max_val - min_val)
    return 1 - normed if invert else normed


def load_silver(session):
    df_listings = session.sql('''
        SELECT id, host_id, neighbourhood_cleansed, property_type,
               room_type, price, availability_365, estimated_occupancy,
               estimated_revenue, number_of_reviews, review_scores_rating,
               review_scores_cleanliness, review_scores_location,
               review_scores_checkin, review_scores_communication,
               review_scores_value
        FROM AIRBNB_INVESTMENT_DB.SILVER.BRISTOL_LISTINGS_SILVER
    ''').to_pandas()

    df_hosts = session.sql('''
        SELECT host_id, host_response_rate, host_acceptance_rate,
               host_is_superhost
        FROM AIRBNB_INVESTMENT_DB.SILVER.BRISTOL_HOSTS_SILVER
    ''').to_pandas()

    df = df_listings.merge(df_hosts, on='host_id', how='left')

    df['host_response_rate'] = pd.to_numeric(
        df['host_response_rate'].str.replace('%', '', regex=False), errors='coerce'
    )
    df['host_acceptance_rate'] = pd.to_numeric(
        df['host_acceptance_rate'].str.replace('%', '', regex=False), errors='coerce'
    )

    print(f'Loaded {len(df)} listings from Silver')
    return df


def compute_metrics(df):
    df = df.copy()
    df['is_entire_home']    = (df['room_type'] == 'Entire home/apt').astype(int)
    df['is_private_room']   = (df['room_type'] == 'Private room').astype(int)
    df['is_shared_room']    = (df['room_type'] == 'Shared room').astype(int)
    df['is_hotel_room']     = (df['room_type'] == 'Hotel room').astype(int)
    df['is_superhost_flag'] = (df['host_is_superhost'] == 't').astype(float)

    df_metrics = df.groupby('neighbourhood_cleansed').agg(
        listing_count            = ('id', 'count'),
        total_reviews            = ('number_of_reviews', 'sum'),
        avg_price                = ('price', 'mean'),
        min_price                = ('price', 'min'),
        max_price                = ('price', 'max'),
        avg_occupancy            = ('estimated_occupancy', 'mean'),
        avg_revenue              = ('estimated_revenue', 'mean'),
        total_revenue            = ('estimated_revenue', 'sum'),
        avg_review_rating        = ('review_scores_rating', 'mean'),
        avg_cleanliness_score    = ('review_scores_cleanliness', 'mean'),
        avg_location_score       = ('review_scores_location', 'mean'),
        avg_checkin_score        = ('review_scores_checkin', 'mean'),
        avg_communication_score  = ('review_scores_communication', 'mean'),
        avg_value_score          = ('review_scores_value', 'mean'),
        room_type_variety        = ('room_type', 'nunique'),
        property_type_variety    = ('property_type', 'nunique'),
        count_entire_home_apt    = ('is_entire_home', 'sum'),
        count_private_room       = ('is_private_room', 'sum'),
        count_shared_room        = ('is_shared_room', 'sum'),
        count_hotel_room         = ('is_hotel_room', 'sum'),
        avg_host_response_rate   = ('host_response_rate', 'mean'),
        avg_host_acceptance_rate = ('host_acceptance_rate', 'mean'),
        pct_superhost            = ('is_superhost_flag', lambda x: x.mean() * 100),
    ).reset_index()

    float_cols = df_metrics.select_dtypes(include=['float64']).columns
    df_metrics[float_cols] = df_metrics[float_cols].round(4)

    print(f'Computed metrics for {len(df_metrics)} neighbourhoods')
    return df_metrics


def compute_persona_scores(df):
    df = df.copy()

    norm_price_inverted = minmax_norm(df['avg_price'], invert=True)
    norm_avg_price      = minmax_norm(df['avg_price'])
    norm_avg_occupancy  = minmax_norm(df['avg_occupancy'])
    norm_avg_revenue    = minmax_norm(df['avg_revenue'])
    norm_avg_rating     = minmax_norm(df['avg_review_rating'])

    df['score_first_time_buyer'] = (
        (norm_price_inverted * 0.35 +
         norm_avg_rating     * 0.35 +
         norm_avg_occupancy  * 0.20 +
         norm_avg_revenue    * 0.10) * 100
    ).round(2)

    df['score_experienced_landlord'] = (
        (norm_avg_revenue   * 0.40 +
         norm_avg_occupancy * 0.35 +
         norm_avg_price     * 0.15 +
         norm_avg_rating    * 0.10) * 100
    ).round(2)

    df['score_passive_income'] = (
        (norm_avg_occupancy * 0.40 +
         norm_avg_revenue   * 0.30 +
         norm_avg_rating    * 0.20 +
         norm_avg_price     * 0.10) * 100
    ).round(2)

    df['computed_at'] = pd.Timestamp.now()

    print(f'Persona scores computed for {len(df)} neighbourhoods')
    return df


def write_gold(session, df):
    session.write_pandas(
        df,
        'BOROUGH_SUMMARY',
        database='AIRBNB_INVESTMENT_DB',
        schema='GOLD',
        overwrite=True
    )
    print(f'Written {len(df)} rows to GOLD.BOROUGH_SUMMARY')


def validate(session):
    df_val = session.sql('''
        SELECT
            COUNT(*) AS neighbourhood_count,
            ROUND(AVG(avg_price), 2) AS mean_avg_price,
            ROUND(MIN(score_first_time_buyer), 2) AS min_ftb_score,
            ROUND(MAX(score_first_time_buyer), 2) AS max_ftb_score,
            ROUND(MIN(score_experienced_landlord), 2) AS min_el_score,
            ROUND(MAX(score_experienced_landlord), 2) AS max_el_score,
            ROUND(MIN(score_passive_income), 2) AS min_pi_score,
            ROUND(MAX(score_passive_income), 2) AS max_pi_score
        FROM AIRBNB_INVESTMENT_DB.GOLD.BOROUGH_SUMMARY
    ''').to_pandas()
    print(df_val)

In [ ]:
# --- Orchestration ---
df_silver  = load_silver(session)
df_metrics = compute_metrics(df_silver)
df_scored  = compute_persona_scores(df_metrics)
write_gold(session, df_scored)
validate(session)

In [ ]:
df_val = session.sql('''
    SELECT
        COUNT(*) AS neighbourhood_count,
        ROUND(AVG(avg_price), 2) AS mean_avg_price,
        ROUND(MIN(score_first_time_buyer), 2) AS min_ftb_score,
        ROUND(MAX(score_first_time_buyer), 2) AS max_ftb_score,
        ROUND(MIN(score_experienced_landlord), 2) AS min_el_score,
        ROUND(MAX(score_experienced_landlord), 2) AS max_el_score,
        ROUND(MIN(score_passive_income), 2) AS min_pi_score,
        ROUND(MAX(score_passive_income), 2) AS max_pi_score
    FROM AIRBNB_INVESTMENT_DB.GOLD.BOROUGH_SUMMARY
''').to_pandas()
print(df_val)

## Borough Summary Complete

`GOLD.BOROUGH_SUMMARY` is ready for Streamlit consumption.
Each row represents one Bristol neighbourhood with persona investment scores.